# The Sonification Encoder — Predictions

**Pre-registration date:** 2026-07-29
**Rule:** All predictions entered here BEFORE the engine is run. Results —
confirmed or failed — remain in the record. Failed predictions stay in the
data, period, full stop.

Two of the four predictions below are the module's **own documented claims**,
registered here so that the record shows whether the documentation is accurate.
They are expected to be tested, not flattered.

---

## P1 — Every frequency is an exact rational ratio of A440

**Claim.** For all 30 entries of `sn.FREQ`, the value is a `fractions.Fraction`,
and `value / 440` is an exact rational just-intonation ratio. No entry is
irrational, and no entry is a float.

**Falsification.** Any entry that is not a `Fraction`, or whose ratio to A440
has a denominator that does not divide a small power of 2 times 3 times 5
(i.e. is not a just ratio).

**Test.** Type check plus exact `Fraction` comparison. No float tolerance is
used anywhere in this test.

## P2 — The `ω = 2πf` round trip costs at most 2 ulp

**Claim.** Converting an exact frequency to `ω = 2πf` in float64 and back by
`ω/(2π)` returns the original float within 2 units in the last place.

**Falsification.** Any tone exceeding 2 ulp.

**Test.** All 30 tones, signed ulp difference.

## P3 — The map is injective

**Claim (the module's implicit claim as an encoder).** Thirty distinct named
symbols receive thirty distinct frequencies, so a decoder can recover the symbol
from the tone.

**Falsification.** Any two distinct names sharing a frequency.

**Test.** Count distinct values; enumerate every collision class.

## P4 — Rest durations are exact integer sample counts

**Claim (quoted from `sonification/tools.py`, confidence ESTABLISHED):**
*"rest durations in samples, exact integer arithmetic"*.

Taken at its word: each of the six `QUASIPARTICLE_RESTS` equals its intended
rational duration exactly, with no remainder discarded.

**Falsification.** Any duration whose intended value is not an integer, i.e.
where `//` discards a non-zero fraction of a sample.

**Test.** Recompute each intended duration as an exact `Fraction` of
`BEAT_SAMPLES` and compare against the stored integer.

---

## Registered as executable assertions

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta.modules.sonification import maths as sn
from ValaQuenta.modules.sonification import SonificationModule

import math
from fractions import Fraction
from collections import defaultdict

print('engine :', 'ValaQuenta/modules/sonification')
print('python :', sys.version.split()[0])
print('A =', sn.CONCERT_A, 'Hz   sample rate =', sn.SAMPLE_RATE,
      '  beat =', sn.BEAT_SAMPLES, 'samples')

In [ ]:
def P1_exact_ratios():
    for name, f in sn.FREQ.items():
        if not isinstance(f, Fraction):
            return False
        ratio = Fraction(f, sn.CONCERT_A)
        # A just ratio has a 5-smooth denominator (2,3,5 only).
        d = ratio.denominator
        for p in (2, 3, 5):
            while d % p == 0:
                d //= p
        if d != 1:
            return False
    return True


def _ulp(got, exact):
    return 0.0 if got == exact else (got - exact)/math.ulp(exact)


def P2_omega_roundtrip(max_ulp=2.0):
    for f in sn.FREQ.values():
        x = float(f)
        back = (2*math.pi*x)/(2*math.pi)
        if abs(_ulp(back, x)) > max_ulp:
            return False
    return True


def P3_injective():
    return len(set(sn.FREQ.values())) == len(sn.FREQ)


def P4_rests_exact():
    B = sn.BEAT_SAMPLES
    intended = {
        'phonon':   Fraction(B, 4),
        'exciton':  Fraction(B, 2),
        'magnon':   Fraction(B*3, 8),
        'roton':    Fraction(B*3, 4),
        'plasmon':  Fraction(B),
        'gravinon': Fraction(B*sn.FIB_NUM, sn.FIB_DEN),
    }
    for name, want in intended.items():
        if want.denominator != 1:
            return False
        if sn.QUASIPARTICLE_RESTS[name] != want:
            return False
    return True


PREDICTIONS = [
    ('P1', 'All 30 tones exact Fraction just ratios of A440', P1_exact_ratios),
    ('P2', 'omega = 2*pi*f round trip within 2 ulp',          P2_omega_roundtrip),
    ('P3', 'The map is injective (30 names -> 30 tones)',     P3_injective),
    ('P4', 'Rest durations exact, as tools.py states',        P4_rests_exact),
]

print(f'{len(PREDICTIONS)} predictions registered, none evaluated yet:')
for tag, desc, _ in PREDICTIONS:
    print(f'  {tag}  {desc}')